# data_processor.py — Test Notebook

Tests the full data pipeline for all 8 supported datasets in three stages:
`_build_messages` (raw image loading) → `__getitem__` (tokenization) → `FlattenedDataCollator` (batching).

---

## Table of Contents

| Section | What it covers |
|---------|----------------|
| **1 — Setup** | sys.path, dataset paths, processor, `LazySupervisedDataset`, dataset index map |
| **2 — R2R baseline** | `__getitem__`, `_build_messages`, `FlattenedDataCollator` on an R2R sample |
| **3 — Video-ChatGPT** | `{video_id, q, a}` format — video sampled from .mp4 |
| **4 — EnvDrop** | `{video_id, instruction}` format — trajectory description task |
| **5 — ShareGPTVideo** | `{video, conversations}` format — frames from directory |
| **6 — ShareGPT4V** | `{image, conversations}` format — multi-turn image QA |
| **7 — ShareGPT `__getitem__`** | tokenization check for both ShareGPT formats |

---

## Supported dataset formats

| Dataset | Annotation keys | Visual input | Notes |
|---------|----------------|--------------|-------|
| r2r / rxr / human | `frames, q, a` | 7 hist + 1 current frame (jpg) | Navigation action prediction |
| scanqa / video_chatgpt | `video_id, q, a` | 8 frames from .mp4 | Video QA |
| envdrop | `video_id, instruction` | 8 frames from .mp4 | Trajectory description |
| sharegptvideo | `video, conversations` | 8 frames from directory | Multi-turn video caption |
| sharegpt4v | `image, conversations` | single image | Multi-turn image QA |

---

> **Quick start:** Run all cells top-to-bottom. To inspect a different sample, change `INDEX` in Section 2 or use `DS_START["<dataset>"]` as the index.

## Section 1 — Setup

Add project root and `qwen-vl-finetune` to `sys.path` so `qwenvl.*` imports resolve from any working directory.

In [1]:
import sys
from pathlib import Path

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "qwen-vl-finetune"))
print(f"Project root: {project_root}")

Project root: /home/chang/Projects/vla_proj/Qwen3-VL


Auto-detect `NAVILA_BASE` from known mount points, define `DATASET_PATHS` for all 8 datasets, and create `MockDataArgs` (mirrors real training args — pixel budgets, frame counts, etc.).

In [2]:
import os
from dataclasses import dataclass

# ---------------------------------------------------------------------------
# Dataset switcher — change DATASET_USE to any key or comma-separated keys
# ---------------------------------------------------------------------------
_NAVILA_CANDIDATES = [
    "/opt/IROS_proj/NaVILA-Dataset",
    "/home/rithvik/IROS_proj/NaVILA-Dataset",
    "/weka/scratch/tinoosh/iros_dataset/NaVILA-Dataset",
]
NAVILA_BASE = next((c for c in _NAVILA_CANDIDATES if os.path.exists(c)), _NAVILA_CANDIDATES[0])
print(f"NAVILA_BASE: {NAVILA_BASE}")

DATASET_PATHS = {
    "r2r":           f"{NAVILA_BASE}/R2R/annotations.json",
    "human":         f"{NAVILA_BASE}/Human/annotations.json",
    "rxr":           f"{NAVILA_BASE}/RxR/annotations.json",
    "scanqa":        f"{NAVILA_BASE}/ScanQA/annotations/ScanQA_v1.0_train_reformat.json",
    "envdrop":       f"{NAVILA_BASE}/EnvDrop/annotations.json",
    "video_chatgpt": f"{NAVILA_BASE}/Video-ChatGPT/VideoInstruct100K.json",
    "sharegptvideo":  f"{NAVILA_BASE}/ShareGPTVideo/video_caption_300k.jsonl",
    "sharegpt4v":     f"{NAVILA_BASE}/ShareGPT4V/sharegpt4v_mix665k_cap23k_coco-ap9k_lcs3k_sam9k_div2k.json",
}

DATASET_USE = "r2r,human,rxr,scanqa,envdrop,video_chatgpt,sharegptvideo,sharegpt4v"
MODEL_TYPE  = "qwen3vl"  # <-- "qwen2vl" | "qwen2.5vl" | "qwen3vl"

print("\nAvailable datasets:")
for name, path in DATASET_PATHS.items():
    exists = os.path.exists(path)
    marker = "OK     " if exists else "MISSING"
    active = "  <-- selected" if name in DATASET_USE.split(",") else ""
    print(f"  [{marker}] {name:15s}  {path}{active}")

@dataclass
class MockDataArgs:
    dataset_use: str = DATASET_USE
    data_flatten: bool = False
    data_packing: bool = False
    model_type: str = MODEL_TYPE
    max_pixels: int = 28 * 28 * 576
    min_pixels: int = 28 * 28 * 16
    video_max_frames: int = 8
    video_min_frames: int = 4
    video_max_pixels: int = 1024 * 28 * 28
    video_min_pixels: int = 256 * 28 * 28
    video_max_total_pixels: int = 1664 * 28 * 28
    video_min_total_pixels: int = 256 * 28 * 28
    video_fps: float = 2.0
    motion_token_text: str = "<motion>"
    gru_history_slots: int = 8
    gru_max_insert_tokens: int = 64
    gru_min_seq_len: int = 1
    gru_fallback_to_stop: bool = True
    base_interval: int = 2
    debug_motion_tokenization: bool = False

data_args = MockDataArgs()
print(f"\ndataset_use: {data_args.dataset_use}")
print(f"model_type:  {data_args.model_type}")

NAVILA_BASE: /opt/IROS_proj/NaVILA-Dataset

Available datasets:
  [OK     ] r2r              /opt/IROS_proj/NaVILA-Dataset/R2R/annotations.json  <-- selected
  [OK     ] human            /opt/IROS_proj/NaVILA-Dataset/Human/annotations.json  <-- selected
  [OK     ] rxr              /opt/IROS_proj/NaVILA-Dataset/RxR/annotations.json  <-- selected
  [OK     ] scanqa           /opt/IROS_proj/NaVILA-Dataset/ScanQA/annotations/ScanQA_v1.0_train_reformat.json  <-- selected
  [OK     ] envdrop          /opt/IROS_proj/NaVILA-Dataset/EnvDrop/annotations.json  <-- selected
  [OK     ] video_chatgpt    /opt/IROS_proj/NaVILA-Dataset/Video-ChatGPT/VideoInstruct100K.json  <-- selected
  [OK     ] sharegptvideo    /opt/IROS_proj/NaVILA-Dataset/ShareGPTVideo/video_caption_300k.jsonl  <-- selected
  [OK     ] sharegpt4v       /opt/IROS_proj/NaVILA-Dataset/ShareGPT4V/sharegpt4v_mix665k_cap23k_coco-ap9k_lcs3k_sam9k_div2k.json  <-- selected

dataset_use: r2r,human,rxr,scanqa,envdrop,video_chatgpt,sharegpt

Load `AutoProcessor` from the local model checkpoint. Provides the tokenizer and image processor used by `apply_chat_template` and pixel encoding downstream.

In [3]:
from transformers import AutoProcessor

MODEL_PATH = "/home/chang/vla/navilla/models/qwen3vl-sft-20260420-1813"
# MODEL_PATH = "/home/chang/vla/navilla/models/qwen3vl-sft-20260420-1813/checkpoint-8000"
# MODEL_PATH = "/home/rithvik/NaVILA-Bench/Qwen-Model/sft_base_qwen3vl"  # rithvik's machine only

processor = AutoProcessor.from_pretrained(MODEL_PATH)
print(f"Tokenizer vocab size: {processor.tokenizer.vocab_size}")
print(f"Image processor type: {type(processor.image_processor).__name__}")

/home/chang/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenizer vocab size: 151643
Image processor type: Qwen2VLImageProcessor


Instantiate `LazySupervisedDataset` with all 8 datasets merged. Annotations are read into `list_data_dict` here; actual image I/O is deferred to `__getitem__`. Expected total: **2,567,704** samples.

In [4]:
from qwenvl.data.data_processor import LazySupervisedDataset

dataset = LazySupervisedDataset(processor, data_args)
print(f"Total samples: {len(dataset)}")

Total samples: 2567704


Single-pass scan of `list_data_dict` to find the starting index of each dataset. Stored in `DS_START["<name>"]` and reused by every test cell below.

In [5]:
DS_START = {}
for i, ann in enumerate(dataset.list_data_dict):
    dp = ann.get("data_path", "")
    for tag in ("R2R", "Human", "RxR", "ScanQA", "EnvDrop",
                "Video-ChatGPT", "ShareGPTVideo", "ShareGPT4V"):
        key = tag.lower().replace("-", "")
        if tag in dp and key not in DS_START:
            DS_START[key] = i
    if len(DS_START) == 8:
        break

print("Dataset start indices:")
for k, v in DS_START.items():
    print(f"  {k:20s}  idx={v}  sample_keys={list(dataset.list_data_dict[v].keys())}")

Dataset start indices:
  r2r                   idx=0  sample_keys=['video_id', 'q', 'a', 'frames', 'data_path']
  human                 idx=353894  sample_keys=['video_id', 'q', 'a', 'frames', 'data_path']
  rxr                   idx=911440  sample_keys=['video_id', 'q', 'a', 'frames', 'data_path']
  scanqa                idx=1329018  sample_keys=['video_id', 'question_id', 'q', 'a', 'object_ids', 'object_names', 'data_path']
  envdrop               idx=1354581  sample_keys=['video_id', 'instruction', 'data_path']
  videochatgpt          idx=1500885  sample_keys=['q', 'a', 'video_id', 'data_path']
  sharegptvideo         idx=1600895  sample_keys=['id', 'video', 'conversations', 'data_path']
  sharegpt4v            idx=1902646  sample_keys=['id', 'image', 'conversations', 'data_path']


## Section 2 — R2R Baseline

`dataset[INDEX]` runs the full pipeline: `_build_messages` → `apply_chat_template` → label masking. Prints tensor shapes, label ratio, the full tokenized sequence structure (image patch counts), and the decoded label.

In [6]:
import torch
import numpy as np

# ── change INDEX to inspect a different sample ──────────────────────────────
INDEX = 0
# ────────────────────────────────────────────────────────────────────────────

sample    = dataset[INDEX]
tokenizer = processor.tokenizer
input_ids = sample["input_ids"][0].tolist()
labels    = sample["labels"]

print("=== __getitem__ output ===\n")
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")

seq_len = sample["attention_mask"][0]
print(f"\nattention_mask = [{seq_len}]  (not a bool mask — collator converts to cu_seqlens [0, {seq_len}])")

total  = labels.numel()
active = (labels != -100).sum().item()
print(f"\nlabels: {active}/{total} tokens active ({active/total:.1%})  — rest ignored (-100)")

IMAGE_PAD = tokenizer.convert_tokens_to_ids("<|image_pad|>")
VIS_START = tokenizer.convert_tokens_to_ids("<|vision_start|>")
IM_START  = tokenizer.convert_tokens_to_ids("<|im_start|>")
IM_END    = tokenizer.convert_tokens_to_ids("<|im_end|>")

segments = []
i = 0
while i < len(input_ids):
    tid = input_ids[i]
    if tid == VIS_START:
        j = i + 1
        while j < len(input_ids) and input_ids[j] == IMAGE_PAD:
            j += 1
        segments.append(f"[IMAGE: {j - i - 1} patches]")
        i = j + 1
    elif tid in (IM_START, IM_END):
        segments.append(tokenizer.decode([tid]))
        i += 1
    else:
        j = i
        while j < len(input_ids) and input_ids[j] not in (VIS_START, IMAGE_PAD, IM_START, IM_END):
            j += 1
        text = tokenizer.decode(input_ids[i:j], skip_special_tokens=True).strip()
        if text:
            segments.append(f'"{text}"')
        i = j

print("\n--- Sequence structure (text collapsed, images shown as patch counts) ---")
for seg in segments:
    print(" ", seg)

label_ids = [tid if tid != -100 else tokenizer.pad_token_id for tid in labels[0].tolist()]
print("\n--- LABEL ---")
print(tokenizer.decode(label_ids, skip_special_tokens=True))

=== __getitem__ output ===

  input_ids            shape=[1, 559]  dtype=torch.int64
  attention_mask       [559]
  mm_token_type_ids    shape=[1, 559]  dtype=torch.int64
  pixel_values         shape=[1568, 1536]  dtype=torch.float32
  image_grid_thw       shape=[8, 3]  dtype=torch.int64
  labels               shape=[1, 559]  dtype=torch.int64
  position_ids         shape=[3, 1, 559]  dtype=torch.int64

attention_mask = [559]  (not a bool mask — collator converts to cu_seqlens [0, 559])

labels: 13/559 tokens active (2.3%)  — rest ignored (-100)

--- Sequence structure (text collapsed, images shown as patch counts) ---
  <|im_start|>
  "system
You are a helpful navigation assistant."
  <|im_end|>
  <|im_start|>
  "user
Imagine you are a robot programmed for navigation tasks. You have been given a video of historical observations:"
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  [IMAGE: 49 patches]
  "

Call `_build_messages` directly on the same R2R annotation to inspect the pre-tokenization message list — PIL image sizes, mode, and exact prompt text.

In [7]:
from qwenvl.data.data_processor import _build_messages
from pathlib import Path

ann       = dataset.list_data_dict[INDEX]
base_path = Path(ann["data_path"])
messages, has_missing = _build_messages(ann, base_path)

print(f"has_missing: {has_missing}  |  turns: {len(messages)}\n")
for turn in messages:
    print(f"[{turn['role']}]")
    for item in turn["content"]:
        if item["type"] == "text":
            print(f"  text : {item['text'][:120]}")
        else:
            img = item["image"]
            print(f"  image: {img.size}  {img.mode}  {type(img).__name__}")
    print()

has_missing: False  |  turns: 3

[system]
  text : You are a helpful navigation assistant.

[user]
  text : Imagine you are a robot programmed for navigation tasks. You have been given a video of historical observations:
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  image: (512, 512)  RGB  Image
  text : and current observation:
  image: (512, 512)  RGB  Image
  text : Your assigned task is: Turn and walk away from the bathroom into the bedroom area. Walk through the open door on the oth

[assistant]
  text : The next action is move forward 75 cm.



Pack two R2R samples with `FlattenedDataCollatorForSupervisedDataset`. Verifies that `attention_mask` is converted from per-sample lengths to cumulative `cu_seqlens` (required by FlashAttention-2 varlen mode).

In [8]:
from qwenvl.data.data_processor import FlattenedDataCollatorForSupervisedDataset

instances = [dataset[INDEX], dataset[INDEX + 1]]

print("=== Before collation (per-sample) ===")
for idx, inst in enumerate(instances):
    seq = inst["attention_mask"][0]
    print(f"\n  sample {idx}:")
    print(f"    input_ids    {list(inst['input_ids'].shape)}")
    print(f"    labels       {list(inst['labels'].shape)}")
    print(f"    position_ids {list(inst['position_ids'].shape)}")
    print(f"    pixel_values {list(inst['pixel_values'].shape)}")
    print(f"    attention_mask (seq len) = {seq}")

collator = FlattenedDataCollatorForSupervisedDataset(tokenizer=processor.tokenizer)
batch = collator(instances)

print("\n=== After collation (batch) ===\n")
for k, v in batch.items():
    if v is None:
        print(f"  {k:20s} None")
    elif hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")

lens = [inst["attention_mask"][0] for inst in instances]
print(f"\nper-sample lengths : {lens}")
print(f"cu_seqlens         : {batch['attention_mask'].tolist()}")
print(f"  → FlashAttention sees 2 independent sequences inside one packed tensor")

=== Before collation (per-sample) ===

  sample 0:
    input_ids    [1, 559]
    labels       [1, 559]
    position_ids [3, 1, 559]
    pixel_values [1568, 1536]
    attention_mask (seq len) = 559

  sample 1:
    input_ids    [1, 523]
    labels       [1, 523]
    position_ids [3, 1, 523]
    pixel_values [1568, 1536]
    attention_mask (seq len) = 523

=== After collation (batch) ===

  input_ids            shape=[1, 1082]  dtype=torch.int64
  labels               shape=[1, 1082]  dtype=torch.int64
  attention_mask       shape=[3]  dtype=torch.int32
  position_ids         shape=[3, 1, 1082]  dtype=torch.int64
  pixel_values         shape=[3136, 1536]  dtype=torch.float32
  image_grid_thw       shape=[16, 3]  dtype=torch.int64
  pixel_values_videos  None
  video_grid_thw       None

per-sample lengths : [559, 523]
cu_seqlens         : [0, 559, 1082]
  → FlashAttention sees 2 independent sequences inside one packed tensor


## Section 3 — Video-ChatGPT

Format `{video_id, q, a}` — same code path as ScanQA. `_build_messages` uniformly samples 8 frames from the `.mp4`. If the video file is missing on disk, `has_missing=True` and the sample contributes zero loss.

In [9]:
vc_ann = dataset.list_data_dict[DS_START["videochatgpt"]]
print(f"Video-ChatGPT keys: {list(vc_ann.keys())}")
print(f"  video_id={vc_ann['video_id']}  q={vc_ann['q'][:60]!r}")

messages, has_missing = _build_messages(vc_ann, Path(vc_ann["data_path"]))
print(f"\nhas_missing: {has_missing}  |  turns: {len(messages)}\n")
for turn in messages:
    print(f"[{turn['role']}]")
    for c in turn["content"]:
        if c["type"] == "text":
            print(f"  text : {c['text'][:100]}")
        else:
            print(f"  image: {c['image'].size}  {c['image'].mode}")
    print()

Video-ChatGPT keys: ['q', 'a', 'video_id', 'data_path']
  video_id=v_k_ZXmr8pmrs  q='What are the main activities that take place in the video?'

has_missing: True  |  turns: 3

[system]
  text : You are a helpful navigation assistant.

[user]
  image: (224, 224)  RGB
  image: (224, 224)  RGB
  image: (224, 224)  RGB
  image: (224, 224)  RGB
  image: (224, 224)  RGB
  image: (224, 224)  RGB
  image: (224, 224)  RGB
  image: (224, 224)  RGB
  text : What are the main activities that take place in the video?

[assistant]
  text : The main activities that take place in the video are the preparation of camera equipment by a man, a



`dataset[idx]` for Video-ChatGPT. Note: if the first sample's `.mp4` is missing on disk, `labels active = 0%` is expected (sample is masked out).

In [10]:
vc_item = dataset[DS_START["videochatgpt"]]

print("=== Video-ChatGPT __getitem__ ===")
for k, v in vc_item.items():
    if hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}")

total  = vc_item["labels"].numel()
active = (vc_item["labels"] != -100).sum().item()
print(f"\nlabels active: {active}/{total} ({active/total:.1%})")
label_ids = [t if t != -100 else processor.tokenizer.pad_token_id for t in vc_item["labels"][0].tolist()]
print(f"label text  : {processor.tokenizer.decode(label_ids, skip_special_tokens=True).strip()!r}")

=== Video-ChatGPT __getitem__ ===
  input_ids            shape=[1, 479]
  mm_token_type_ids    shape=[1, 479]
  pixel_values         shape=[1568, 1536]
  image_grid_thw       shape=[8, 3]
  labels               shape=[1, 479]
  position_ids         shape=[3, 1, 479]

labels active: 0/479 (0.0%)
label text  : ''


## Section 4 — EnvDrop

Format `{video_id (int), instruction}`. Video file = `{video_id}.mp4` (range 1–146413). Prompt asks the model to describe the navigation trajectory; target answer is the `instruction` field.

In [11]:
ed_ann = dataset.list_data_dict[DS_START["envdrop"]]
print(f"EnvDrop keys: {list(ed_ann.keys())}")
print(f"  video_id={ed_ann['video_id']}  instruction={ed_ann['instruction'][:60]!r}")

messages, has_missing = _build_messages(ed_ann, Path(ed_ann["data_path"]))
print(f"\nhas_missing: {has_missing}  |  turns: {len(messages)}\n")
for turn in messages:
    print(f"[{turn['role']}]")
    for c in turn["content"]:
        if c["type"] == "text":
            print(f"  text : {c['text'][:100]}")
        else:
            print(f"  image: {c['image'].size}  {c['image'].mode}")
    print()

EnvDrop keys: ['video_id', 'instruction', 'data_path']
  video_id=1  instruction='Walk past the table and chairs and turn right. Walk up the s'

has_missing: False  |  turns: 3

[system]
  text : You are a helpful navigation assistant.

[user]
  text : Assume you are a robot designed for navigation. You are provided with captured images sequences
  image: (512, 512)  RGB
  image: (512, 512)  RGB
  image: (512, 512)  RGB
  image: (512, 512)  RGB
  image: (512, 512)  RGB
  image: (512, 512)  RGB
  image: (512, 512)  RGB
  image: (512, 512)  RGB
  text : . Based on this image sequence, please describe the navigation trajectory of the robot.

[assistant]
  text : Walk past the table and chairs and turn right. Walk up the stairs and stop at the top of the stairs.



`dataset[idx]` for EnvDrop — confirms that label text decodes back to the original `instruction` string.

In [12]:
ed_item = dataset[DS_START["envdrop"]]

print("=== EnvDrop __getitem__ ===")
for k, v in ed_item.items():
    if hasattr(v, "shape"):
        print(f"  {k:20s} shape={list(v.shape)}")

total  = ed_item["labels"].numel()
active = (ed_item["labels"] != -100).sum().item()
print(f"\nlabels active: {active}/{total} ({active/total:.1%})")
label_ids = [t if t != -100 else processor.tokenizer.pad_token_id for t in ed_item["labels"][0].tolist()]
print(f"label text  : {processor.tokenizer.decode(label_ids, skip_special_tokens=True).strip()!r}")

=== EnvDrop __getitem__ ===
  input_ids            shape=[1, 486]
  mm_token_type_ids    shape=[1, 486]
  pixel_values         shape=[1568, 1536]
  image_grid_thw       shape=[8, 3]
  labels               shape=[1, 486]
  position_ids         shape=[3, 1, 486]

labels active: 25/486 (5.1%)
label text  : 'Walk past the table and chairs and turn right. Walk up the stairs and stop at the top of the stairs.'


## Section 5 — ShareGPTVideo

Format `{id, video, conversations}`. Frames are pre-extracted in `frames/{video}/` as `.jpeg`. 8 frames sampled uniformly. `<video>` tokens stripped from conversation and replaced by image content items.

In [13]:
sgv_ann = dataset.list_data_dict[DS_START["sharegptvideo"]]
print(f"ShareGPTVideo keys: {list(sgv_ann.keys())}")
print(f"  video={sgv_ann['video']}  turns={len(sgv_ann['conversations'])}")

messages, has_missing = _build_messages(sgv_ann, Path(sgv_ann["data_path"]))
print(f"\nhas_missing: {has_missing}  |  turns: {len(messages)}\n")
for turn in messages:
    print(f"[{turn['role']}]")
    for c in turn["content"]:
        if c["type"] == "text":
            print(f"  text : {c['text'][:100]}")
        else:
            print(f"  image: {c['image'].size}  {c['image'].mode}")
    print()

ShareGPTVideo keys: ['id', 'video', 'conversations', 'data_path']
  video=v_--0edUL8zmA-Scene-001  turns=2

has_missing: False  |  turns: 3

[system]
  text : You are a helpful navigation assistant.

[user]
  image: (336, 252)  RGB
  image: (336, 252)  RGB
  image: (336, 252)  RGB
  image: (336, 252)  RGB
  image: (336, 252)  RGB
  image: (336, 252)  RGB
  image: (336, 252)  RGB
  image: (336, 252)  RGB
  text : Write an in-depth depiction of the video, covering all its aspects.

[assistant]
  text : The video opens with a view of an indoor court marked with various lines, presumably for different s



## Section 6 — ShareGPT4V

Format `{id, image, conversations}`. Single image, multi-turn QA. Image prepended to first user turn only; subsequent turns are text-only. Label masking covers every assistant segment across all turns.

In [14]:
start = DS_START["sharegpt4v"]
sg4v_ann = next(
    dataset.list_data_dict[i]
    for i in range(start, start + 10000)
    if len(dataset.list_data_dict[i].get("conversations", [])) >= 6
)
n_rounds = len(sg4v_ann["conversations"]) // 2
print(f"ShareGPT4V keys: {list(sg4v_ann.keys())}")
print(f"  id={sg4v_ann['id']}  image={sg4v_ann['image']}  ({n_rounds} rounds)")

messages, has_missing = _build_messages(sg4v_ann, Path(sg4v_ann["data_path"]))
print(f"\nhas_missing: {has_missing}  |  turns: {len(messages)}\n")
for turn in messages:
    print(f"[{turn['role']}]")
    for c in turn["content"]:
        if c["type"] == "text":
            print(f"  text : {c['text'][:100]}")
        else:
            print(f"  image: {c['image'].size}  {c['image'].mode}")
    print()

ShareGPT4V keys: ['id', 'image', 'conversations', 'data_path']
  id=000000033471  image=coco/train2017/000000033471.jpg  (3 rounds)

has_missing: False  |  turns: 7

[system]
  text : You are a helpful navigation assistant.

[user]
  image: (640, 480)  RGB
  text : What are the colors of the bus in the image?

[assistant]
  text : The bus in the image is white and red.

[user]
  text : What feature can be seen on the back of the bus?

[assistant]
  text : The back of the bus features an advertisement.

[user]
  text : Is the bus driving down the street or pulled off to the side?

[assistant]
  text : The bus is driving down the street, which is crowded with people and other vehicles.



## Section 7 — ShareGPT `__getitem__`

Full pipeline test for both ShareGPT formats. Label ratio should be noticeably higher than navigation datasets (~25%) since multi-turn responses are much longer. Decoded label should concatenate all assistant turns.

In [15]:
def check_item(label, idx):
    item = dataset[idx]
    total  = item["labels"].numel()
    active = (item["labels"] != -100).sum().item()
    label_ids = [t if t != -100 else processor.tokenizer.pad_token_id
                 for t in item["labels"][0].tolist()]
    text = processor.tokenizer.decode(label_ids, skip_special_tokens=True).strip()
    print(f"\n=== {label} (idx={idx}) ===")
    for k, v in item.items():
        if hasattr(v, "shape"):
            print(f"  {k:20s} shape={list(v.shape)}")
    print(f"  labels active : {active}/{total} ({active/total:.1%})")
    print(f"  first label   : {text[:120]!r}")

check_item("ShareGPTVideo", DS_START["sharegptvideo"])
check_item("ShareGPT4V",    DS_START["sharegpt4v"])


=== ShareGPTVideo (idx=1600895) ===
  input_ids            shape=[1, 592]
  mm_token_type_ids    shape=[1, 592]
  pixel_values         shape=[1536, 1536]
  image_grid_thw       shape=[8, 3]
  labels               shape=[1, 592]
  position_ids         shape=[3, 1, 592]
  labels active : 158/592 (26.7%)
  first label   : 'The video opens with a view of an indoor court marked with various lines, presumably for different sports. There are fou'

=== ShareGPT4V (idx=1902646) ===
  input_ids            shape=[1, 165]
  mm_token_type_ids    shape=[1, 165]
  pixel_values         shape=[192, 1536]
  image_grid_thw       shape=[1, 3]
  labels               shape=[1, 165]
  position_ids         shape=[3, 1, 165]
  labels active : 42/165 (25.5%)
  first label   : 'The bus in the image is white and red.\nThe back of the bus features an advertisement.\nThe bus is driving down the street'
